# Python Package Management and Publishing with `pip`

This is a **from basics to high intermediate** lesson in **shipping** Python code: moving from “it runs on my laptop” to “someone can `pip install` a **named, versioned** release.” You will use **`pyproject.toml`**, build **wheels** and **sdists**, prove installs in **virtual environments**, rehearse on **TestPyPI**, publish to **PyPI**, and cover **credentials**, **private indexes**, **versioning**, and **life after upload** (yank, ownership, what you cannot undo).

### What you will learn

You should be able to:

- **Name the moving parts** (import package vs distribution, wheel vs sdist, index vs installer) so documentation and error messages make sense.
- **Lay out a project** with **`pyproject.toml`** and a **`src/`** tree in line with current ecosystem defaults.
- **Declare** runtime deps, optional **extras**, and **console scripts** that become shell commands after install.
- **Build** wheels and source distributions and choose the right check for each stage of development.
- **Install locally** in a **clean venv**, including **editable** mode for day-to-day coding and “install from `dist/`” for release-like verification.
- **Publish** using the usual TestPyPI rehearsal and PyPI production flow, and know where **authentication** fits.
- **Bump versions** and avoid fighting PyPI’s **immutability** rules.
- **Point `pip` at other indexes** safely and understand **dependency confusion** in broad terms.
- **Maintain** releases after the fact (yank, ownership, realistic expectations about deletion).

### How to use this notebook

Run cells **in order** when you can. The next cell creates a **temporary workspace** and a tiny demo package so you can run real `pip` and file-tree examples without cluttering this repository. If an optional tool (for example `build`) is missing, the notebook will say so—install it if you want every command to succeed on your machine.

The outline below is your map through the lesson.

### Contents

- [1. Packaging terms: package, distribution, `pip`, PyPI](#1-packaging-terms-package-distribution-pip-pypi)
- [2. Minimal project layout with `pyproject.toml`](#2-minimal-project-layout-with-pyprojecttoml)
- [3. Metadata, dependencies, and entry points](#3-metadata-dependencies-and-entry-points)
- [4. Building wheels and source distributions](#4-building-wheels-and-source-distributions)
- [5. Local installs, editable mode, and virtual environments](#5-local-installs-editable-mode-and-virtual-environments)
- [6. Publishing to TestPyPI and PyPI](#6-publishing-to-testpypi-and-pypi)
- [7. Authentication, authorization, and `.pypirc`](#7-authentication-authorization-and-pypirc)
- [8. Updating releases and versioning](#8-updating-releases-and-versioning)
- [9. Private indexes and dependency sources](#9-private-indexes-and-dependency-sources)
- [10. Managing releases: yank, replace, transfer, remove](#10-managing-releases-yank-replace-transfer-remove)

In [ ]:
from __future__ import annotations

import os, sys, venv, configparser, shutil, \
    subprocess, tempfile, textwrap, tomllib
from pathlib import Path

WORKDIR = Path("./g_pip/")
print("Notebook workspace:", WORKDIR)

class PkgUtils:
    @classmethod
    def run(cls, cmd, cwd = None, env = None):
        print("$", " ".join(map(str, cmd)))
        completed: subprocess.CompletedProcess = subprocess.run(cmd,
           cwd = cwd, env = env, text = True, capture_output = True)
        if completed.stdout.strip(): print(completed.stdout.strip())
        if completed.stderr.strip(): print(completed.stderr.strip())
        print("return code:", completed.returncode)
        return completed

    @classmethod
    def tree(cls, root: Path, max_depth: int = 3) -> None:
        root = Path(root)
        for path in sorted(root.rglob("*")):
            depth = len(path.relative_to(root).parts)
            if (depth > max_depth): continue
            indent = "  " * (depth - 1)
            suffix = "/" if path.is_dir() else ""
            print(f"{indent}{path.name}{suffix}")

    @classmethod
    def demo(cls, root: Path, version: str = "0.1.0") -> Path:
        root = Path(root)
        src = root / "src" / "demo_pkg"
        src.mkdir(parents=True, exist_ok=True)

        (root / "README.md").write_text(
            "# demo-pkg\n\nA tiny package created inside g_pip.ipynb.\n",
            encoding="utf-8",
        )
        (src / "__init__.py").write_text(
            textwrap.dedent(
                f'''
                __all__ = ["__version__", "hello", "meaning_of_packaging"]
                __version__ = "{version}"

                def hello(name: str = "world") -> str:
                    return f"hello, {{name}}"

                def meaning_of_packaging() -> str:
                    return "Package once, install reproducibly."
                '''
            ).strip()
            + "\n",
            encoding="utf-8",
        )
        (src / "cli.py").write_text(
            textwrap.dedent(
                """
                from . import hello

                def main() -> None:
                    print(hello("from console script"))
                """
            ).strip()
            + "\n",
            encoding="utf-8",
        )
        (root / "pyproject.toml").write_text(
            textwrap.dedent(
                f"""
                [build-system]
                requires = ["setuptools>=68", "wheel"]
                build-backend = "setuptools.build_meta"

                [project]
                name = "demo-pkg-cursor"
                version = "{version}"
                description = "Temporary demo package for packaging examples"
                readme = "README.md"
                requires-python = ">=3.10"
                authors = [{{name = "Notebook Demo"}}]
                dependencies = []

                [project.optional-dependencies]
                dev = ["pytest>=8", "ruff>=0.6"]
                docs = ["mkdocs>=1.6"]

                [project.scripts]
                demo-hello = "demo_pkg.cli:main"

                [tool.setuptools]
                package-dir = {{"" = "src"}}

                [tool.setuptools.packages.find]
                where = ["src"]
                """
            ).strip()
            + "\n",
            encoding="utf-8",
        )
        return root

DEMO = PkgUtils.demo(WORKDIR / "demo_pkg_project")
print("Created demo project:")
PkgUtils.tree(DEMO)

Notebook workspace: g_pip
Created demo project:
pyproject.toml
README.md
src/
  demo_pkg/
    __init__.py
    cli.py


## 1. Packaging terms: package, distribution, `pip`, PyPI

Python packaging vocabulary is overloaded: the same word “package” can mean “a folder you `import`” or “something you install from the internet.” Clearing that up early saves hours of confusion when you read PyPI, `pip`, or CI logs.

### Import package vs distribution

When you write `import demo_pkg`, you are loading an **import package**: a module or package tree on Python’s import path. That name is what appears in code.

When someone runs `pip install demo-pkg-cursor`, they are installing a **distribution**: a *named, versioned publishable unit* that carries metadata, dependencies, and (usually) the code that ends up as importable modules. The distribution name is what you see on PyPI and in `pip list`; it often uses hyphens, while the import name may use underscores. They **do not have to match**, though keeping them similar avoids endless “which name do I use where?” questions.

### Installer and index

**`pip`** is the tool that **downloads** distributions, **resolves** compatible versions of dependencies, and **installs** files into an environment. It is not “the registry”; it is the client.

**PyPI** (the Python Package Index) is the **default public catalog** `pip` queries. You can point `pip` at other indexes (TestPyPI, a company mirror, a local directory of wheels). So the accurate mental model is: **indexes host distributions; `pip` installs them.**

People sometimes say “upload to pip.” What actually happens is: you **upload distribution files to an index** (often PyPI), and **`pip install`** fetches from that index.

### Artifacts and releases

Before or instead of publishing, build tools produce files you can install or upload:

- A **wheel** (`.whl`) is a **built** install artifact. Installing it is usually fast and does not require running `setup.py` on the end user’s machine.
- An **sdist** (source distribution, often `.tar.gz`) is a **source** snapshot. The installer or build backend may still need to build a wheel from it on the target platform.

Finally, a **version** string (e.g. `0.1.0`) identifies a **release** of a distribution on the index. On PyPI, a published version is effectively **immutable**—you do not “swap in” a fixed wheel under the same version number. That design protects reproducibility: two people who asked for `mypkg==1.2.3` should get the same logical release.

The next cell prints a small cheat sheet mapping these terms to the demo project.

In [ ]:
print("Python executable:", sys.executable)
run([sys.executable, "-m", "pip", "--version"])

terms = {
    "import package": "demo_pkg",
    "distribution": "demo-pkg-cursor",
    "installer": "pip",
    "index": "PyPI (or another simple index)",
    "built artifact": "wheel (.whl)",
    "source artifact": "sdist (.tar.gz)",
}

for key, value in terms.items():
    print(f"{key:15} -> {value}")

## 2. Minimal project layout with `pyproject.toml`

For many years, Python projects used a mix of `setup.py`, `setup.cfg`, and tool-specific files. Today, **`pyproject.toml`** is the **single, standard place** where you declare: “how do I build this?” and “what is this project, in the eyes of installers and PyPI?”

Think of `pyproject.toml` as a **contract**:

1. **`[build-system]`** tells tools like `pip` which **build backend** to use (`setuptools.build_meta`, `hatchling.build`, etc.) and which packages must be present in an **isolated build environment** to run that backend.
2. **`[project]`** holds **PyPI-facing metadata**: human-readable name, version, supported Python versions, dependencies, optional “extras,” URLs, license expressions, and entry points for console scripts.
3. **Tool tables** (e.g. `[tool.setuptools]`) configure the backend—where packages live, what to include in the sdist, and so on.

You can still find older tutorials centered on `setup.py`. For new work, prefer **`pyproject.toml`** unless you are maintaining legacy code that has not migrated yet.

### Why a `src/` layout?

A **flat layout** puts `mypkg/` next to `pyproject.toml` at the repo root. That works, but it is easy for tests or notebooks to **accidentally import the wrong copy** of the code (the working tree) instead of the **installed** package, hiding packaging bugs until release.

The **`src/` layout** places importable code under `src/mypkg/`. Combined with correct setuptools configuration, it nudges everyone toward: **install the package (even editable) and import what users will get.** That is closer to real-world usage and catches path mistakes earlier.

Typical skeleton:

- `pyproject.toml` — build backend + project metadata
- `README.md` — long description / docs entry point
- `src/demo_pkg/__init__.py` — your actual package
- `src/demo_pkg/...` — submodules, CLI modules, etc.

The following cell shows the demo project tree and the contents of `pyproject.toml` and `__init__.py` so you can connect filenames to concepts.

In [ ]:
print("Project tree:")
show_tree(DEMO)

print("\npyproject.toml:\n")
print((DEMO / "pyproject.toml").read_text(encoding="utf-8"))

print("\nsrc/demo_pkg/__init__.py:\n")
print((DEMO / "src" / "demo_pkg" / "__init__.py").read_text(encoding="utf-8"))

## 3. Metadata, dependencies, and entry points

The **`[project]`** table is what **PyPI**, `pip`, and other installers read to answer user-facing questions: What is this thing called? Which Python versions are supported? What gets installed automatically? What optional bundles exist?

### Core metadata

- **`name`** — The **distribution name** on PyPI and in `pip install ...`. It is normalized (case-insensitive; hyphens vs underscores are treated in a standard way by the ecosystem).
- **`version`** — The **release identifier** for this upload. It should change whenever you publish a new release users are meant to consume.
- **`requires-python`** — The **supported interpreter range** (PEP 508). Installers use this to refuse impossible installs early (for example “this project needs 3.11+”).
- **`readme`**, **`description`**, **`authors`**, URLs, license — Human and legal context. A good README path means PyPI can show rich project pages.

### Dependencies and extras

- **`dependencies`** — **Runtime** requirements: what must be present for `import mypkg` and normal use. Keep this lean; heavy tooling belongs elsewhere.
- **`optional-dependencies`** — Named **extras**, e.g. `pip install "demo-pkg-cursor[dev]"`. Typical groups include `dev` (tests, linters), `docs`, or feature flags like `postgres`. Extras are how you avoid forcing every user to install Jupyter or Sphinx.

### Console scripts (entry points)

**`[project.scripts]`** maps a **command name** to a **callable** inside your package, e.g. `demo-hello = "demo_pkg.cli:main"`. After install, the environment gets a `demo-hello` executable (on Windows, often a wrapper script) that invokes your function. That is how CLI tools become first-class commands without asking users to run `python -m ...`.

### Distribution name vs import name (again)

The **PyPI name** (`demo-pkg-cursor`) and the **import package** (`demo_pkg`) **may differ**. That is legal and sometimes intentional (branding vs module naming). It is also a common source of typos—when in doubt, check the project’s docs for “how to import” versus “how to pip install.”

The next cell parses the demo `pyproject.toml` so you can see these fields as structured data.

In [ ]:
data = tomllib.loads((DEMO / "pyproject.toml").read_text(encoding="utf-8"))
project = data["project"]

print("Distribution name:", project["name"])
print("Version:", project["version"])
print("Requires Python:", project["requires-python"])
print("Dependencies:", project["dependencies"])
print("Optional dependency groups:", list(project["optional-dependencies"].keys()))
print("Console scripts:", project["scripts"])
print("Import package:", "demo_pkg")

## 4. Building wheels and source distributions

Publishing is not “copy my folder to PyPI.” Indexes accept **specific file types** that installers know how to consume. In practice you **build** those files from your source tree, **inspect** them (sizes, filenames, tags), and only then **upload** them.

### What you are producing

- **Wheel (`*.whl`)** — A **built** distribution: metadata plus files laid out for installation. Wheels are usually **fast to install** and avoid running your package’s build logic on every end user’s machine. Platform-specific wheels can include compiled extensions; **pure-Python** wheels are broadly reusable.
- **Source distribution (sdist, often `*.tar.gz`)** — A **source snapshot** plus enough metadata for a build backend to produce a wheel (or a legacy install) **on the target** or in a build environment. Some platforms or policies still expect an sdist; it is also a canonical “here is exactly what we shipped as source” artifact.

Neither file is a substitute for **version control**; both are **release artifacts** derived from a tagged state of your repo.

### Commands you will see in the wild

- **`python -m build --sdist --wheel`** — The dedicated **`build`** project invokes your declared build backend and writes artifacts to `dist/`. This is the clearest “make me both standard formats” entry point when the package is installed.
- **`python -m pip wheel ...`** — Asks **`pip`** to build a wheel, often used in CI caches or when you want wheels without a separate `build` install. Flags like `--no-deps` focus the command on *your* project only.

The next cell runs **`pip wheel`** (always available if `pip` is) and then tries **`python -m build`** if present. Compare the `dist/` listing to the theory above: filenames encode **distribution name, version, and compatibility tags**.

In [ ]:
dist_dir = DEMO / "dist"
shutil.rmtree(dist_dir, ignore_errors=True)

print("Build a wheel with pip:")
run(
    [
        sys.executable,
        "-m",
        "pip",
        "wheel",
        "--no-deps",
        "--no-build-isolation",
        ".",
        "-w",
        "dist",
    ],
    cwd=DEMO,
)

print("\nTry building both wheel and sdist with python -m build:")
build_result = run(
    [sys.executable, "-m", "build", "--sdist", "--wheel", "--outdir", "dist"],
    cwd=DEMO,
)
if build_result.returncode != 0:
    print("`python -m build` is optional. Install it with `python -m pip install build` if you want both sdist and wheel locally.")

print("\nArtifacts in dist/:")
if dist_dir.exists():
    for path in sorted(dist_dir.iterdir()):
        print("-", path.name)
else:
    print("dist/ was not created")

## 5. Local installs, editable mode, and virtual environments

**Global `pip install` on your system Python** is a common way to break tooling or collide versions. For packaging work, treat a **virtual environment** as a disposable lab: create it, install candidates, run imports and CLI entry points, delete it if needed.

### Why `python -m venv`?

A venv gives you an isolated **`site-packages`**, its own **`python`**, and its own **`pip`**. That answers the question: “If a user only had this environment, would `import mypkg` and `my-cli-command` work?” without interference from everything else you have installed.

### Normal vs editable installs

- **`python -m pip install .`** — Copies (or links) the **built** result into the environment. Good for checking what a **non-developer** experiences after a release. You typically rebuild or reinstall after meaningful changes.
- **`python -m pip install -e .` (editable)** — Installs **metadata and entry points** while leaving the **importable code** wired to your working tree. Edit source, run tests or CLIs immediately—ideal for **daily development**. It can still hide packaging mistakes (for example forgetting to declare package data) because your repo layout is on the path.

**Best practice:** use **editable** while coding, but before tagging a release also create a **fresh venv** and install from the **`dist/*.whl`** you intend to upload. That combination catches “works in my checkout” problems early.

The next cell creates a temporary venv under the notebook workspace and demonstrates an **editable** install of the demo project.

In [ ]:
venv_dir = WORKDIR / "venv_demo"
if not venv_dir.exists():
    venv.EnvBuilder(with_pip=True).create(venv_dir)

venv_python = venv_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print("Virtual environment Python:", venv_python)

run([str(venv_python), "-m", "pip", "--version"])
install_result = run(
    [str(venv_python), "-m", "pip", "install", "--no-build-isolation", "-e", str(DEMO)]
)
if install_result.returncode == 0:
    run(
        [
            str(venv_python),
            "-c",
            "import demo_pkg; print(demo_pkg.__version__); print(demo_pkg.hello('editable install'))",
        ]
    )
else:
    print("Editable install failed in this environment; the command above is still the right workflow to test in a clean venv.")

## 6. Publishing to TestPyPI and PyPI

**PyPI** is production: names and versions matter, mistakes are visible, and **you generally cannot rewrite history** for a given version. **TestPyPI** is a separate index with the **same mechanics** but different data—perfect for dry runs.

### A sane first-time flow

1. **Build** wheels (and usually sdists) locally from a clean tree; ensure `dist/` contains only what you intend to ship.
2. **Upload to TestPyPI** with **`twine`** (or another trusted uploader). Fix metadata and filenames until the index page looks right.
3. In a **brand-new venv**, run **`pip install`** with **`--index-url`** pointing at TestPyPI. Because many packages are **not** mirrored on TestPyPI, you often add **`--extra-index-url https://pypi.org/simple`** so *dependencies* resolve from the real PyPI while *your* package comes from the test index.
4. When satisfied, **upload the same versioned artifacts to PyPI** (or bump the version if you changed anything—remember **immutability**).

### Tools and modern auth

- **`twine`** validates and uploads distributions; it is the de facto CLI for maintainers.
- **Trusted publishing** (e.g. GitHub Actions + OIDC) lets PyPI grant **short-lived upload permissions** to a known workflow, reducing reliance on long-lived API tokens stored in secrets.

The recurring idea: **indexes store distributions; `pip` downloads and installs them.** You are not “putting code on pip”—you are **registering releases** with an index `pip` already knows how to talk to.

The next cell prints example commands; adjust project names and index URLs to match your real package.

In [ ]:
run([sys.executable, "-m", "twine", "--version"])

publish_commands = [
    "python -m build --sdist --wheel",
    "python -m twine upload --repository testpypi dist/*",
    "python -m pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple demo-pkg-cursor",
    "python -m twine upload dist/*",
]

print("Recommended publish flow:")
for command in publish_commands:
    print("-", command)

print("\nIf you use GitHub Actions trusted publishing, PyPI can trust the workflow identity instead of a long-lived API token.")

## 7. Authentication, authorization, and `.pypirc`

Uploading to PyPI is a **remote API call** with real consequences: a successful upload creates a **release** others may install. Two separate ideas show up in docs and incident postmortems:

- **Authentication** — *Who is this request from?* Typical answers: a **PyPI API token**, credentials in CI secrets, or a **trusted publishing** identity (OIDC) tied to a known automation job.
- **Authorization** — *What is that identity allowed to do?* Project **owners** and **maintainers**, organization policies, and token **scopes** decide whether you can upload, yank, or manage settings.

### Where secrets should live

- **Environment variables** (e.g. `TWINE_PASSWORD` or tool-specific vars) and **secret managers** are preferred for tokens: they rotate cleanly and never belong in Git.
- **Trusted publishing** reduces long-lived secrets by letting PyPI trust a **short-lived token** issued to a verified CI workload.
- **`.pypirc`** is a local **INI file** many tools read for default upload URLs and credentials. It is convenient on a developer laptop but easy to mis-handle (wrong permissions, accidental copies into backups). Treat it like SSH keys: machine-local, not committed.

### Team reality

Packaging is also **people**: add co-maintainers before the only owner leaves, document who rotates tokens, and rehearse **revocation** if a credential leaks. The next cell writes a **fake** `.pypirc` with obviously bogus tokens—compare its shape to what you would configure locally, then delete real secrets from chat logs and screenshots.

In [ ]:
pypirc_path = WORKDIR / ".pypirc.example"
pypirc_path.write_text(
    textwrap.dedent(
        """
        [distutils]
        index-servers =
            pypi
            testpypi

        [pypi]
        repository = https://upload.pypi.org/legacy/
        username = __token__
        password = pypi-AgEIcHlwaS5vcmcCJDEyMzQ1Njc4OTBhYmNkZWY=

        [testpypi]
        repository = https://test.pypi.org/legacy/
        username = __token__
        password = pypi-AgEIcHlwaS5vcmcCJGFiY2RlZjAxMjM0NTY3ODk=
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

parser = configparser.ConfigParser()
parser.read(pypirc_path, encoding="utf-8")

print("Wrote example config:", pypirc_path)
for repo_name in ("pypi", "testpypi"):
    section = parser[repo_name]
    masked = section["password"][:12] + "..."
    print(repo_name, "->", section["repository"], "as", section["username"], "token", masked)

print("\nReal projects should prefer environment variables or trusted publishing over committing secrets.")

## 8. Updating releases and versioning

PyPI’s immutability rule is a **feature for users**: if I pin `requests==2.31.0`, I expect that pin to keep meaning the **same release** tomorrow. That means **you cannot “fix” a broken upload in place** by replacing files under the same version. The supported story is almost always: **ship a new version** (and optionally **yank** the bad one—covered later).

### Workflow habits that pay off

- **Bump `version` in `pyproject.toml` (or your tool’s equivalent) before every publish** you intend users to consume.
- **Tag the same commit** in Git (`v1.2.3`) so source, CI artifacts, and changelog entries line up.
- **Write release notes**: even a short GitHub Release or `CHANGELOG.md` entry saves downstream debugging.

### SemVer as a shared language

Many libraries follow **semantic versioning** in spirit:

- **MAJOR** — incompatible API or behavior changes consumers must react to.
- **MINOR** — new functionality in a backward-compatible way.
- **PATCH** — bug fixes without intended API churn.

Not every project is strict SemVer, but **never reuse a version number for different bits**: once `1.0.0` means something on the index, let the next story be `1.0.1` or `1.1.0`.

The following cell copies the demo project, bumps the **patch** version with a tiny helper, and prints the Git/build/upload commands you would typically run next.

In [ ]:
import re


def bump_patch(version: str) -> str:
    major, minor, patch = map(int, version.split("."))
    return f"{major}.{minor}.{patch + 1}"


current_text = (DEMO / "pyproject.toml").read_text(encoding="utf-8")
current_version = tomllib.loads(current_text)["project"]["version"]
next_version = bump_patch(current_version)
next_project = WORKDIR / "demo_pkg_project_next"
shutil.rmtree(next_project, ignore_errors=True)
shutil.copytree(DEMO, next_project)

updated_text = re.sub(
    rf'version = "{re.escape(current_version)}"',
    f'version = "{next_version}"',
    current_text,
    count=1,
)
(next_project / "pyproject.toml").write_text(updated_text, encoding="utf-8")

print("Current version:", current_version)
print("Next patch version:", next_version)
print("Updated file:", next_project / "pyproject.toml")
print("\nTypical release commands:")
print(f"- git tag v{next_version}")
print("- python -m build --sdist --wheel")
print("- python -m twine upload dist/*")

## 9. Private indexes and dependency sources

Companies rarely rely on **only** `pypi.org`. Internal libraries live on **private indexes** (Artifactory, Nexus, devpi, a static “simple” index behind VPN). Teams also stash **wheelhouses**—directories or HTTP folders full of `.whl` files—for air-gapped installs or CI speed.

`pip` therefore needs more than “default PyPI”: it needs a **resolution policy** (which index to query first) and sometimes **explicit pins** so yesterday’s deploy matches today’s.

### Flags you will configure

- **`--index-url`** — Primary Simple API index for this command (overrides the default PyPI URL).
- **`--extra-index-url`** — Additional indexes consulted during resolution. Handy for TestPyPI + PyPI or private + public mixes; also the place **dependency confusion** attacks show up if an attacker can squat a name on an index you query.
- **`--find-links`** — Install from a **local path** or URL listing wheels without a full index; great for internal drops.
- **`-r requirements.txt`** — Declarative list of requirements; often paired with **`-c constraints.txt`** to cap transitive versions (`urllib3<3`, `requests!=2.32.0`) without listing every package explicitly.

### Requirements vs constraints

Think of **requirements** as “what I want installed” and **constraints** as “versions allowed anywhere in the graph.” Splitting them helps large apps keep **reproducible** environments while still expressing direct dependencies clearly.

The next cell writes toy `requirements.txt` and `constraints.txt` files and prints example `pip install` lines. It ends with `pip config list` so you can see whether your machine already sets global index URLs.

In [ ]:
requirements_path = WORKDIR / "requirements.txt"
constraints_path = WORKDIR / "constraints.txt"

requirements_path.write_text(
    "demo-pkg-cursor==0.1.0\ninternal-lib>=2.0\n",
    encoding="utf-8",
)
constraints_path.write_text(
    "urllib3<3\nrequests!=2.32.0\n",
    encoding="utf-8",
)

print("requirements.txt:\n")
print(requirements_path.read_text(encoding="utf-8"))
print("constraints.txt:\n")
print(constraints_path.read_text(encoding="utf-8"))

print("Useful install commands:")
print("- python -m pip install -r requirements.txt -c constraints.txt")
print("- python -m pip install --index-url https://pypi.my-company.example/simple internal-lib")
print("- python -m pip install --find-links ./wheelhouse demo-pkg-cursor")
run([sys.executable, "-m", "pip", "config", "list"])

## 10. Managing releases: yank, replace, transfer, remove

Shipping `1.0.0` is not the end state—it is the start of **operations**: users pin versions, CI caches wheels, and security issues appear months later. Knowing what PyPI **can** and **cannot** do keeps incidents calmer.

### Replace, yank, and “delete”

- **Replacing files** — Assume **impossible** for a published version. If the tarball is wrong, you **do not** re-upload `1.2.3` with a fix; you publish **`1.2.4`** (or whatever your scheme dictates).
- **Yanking** — Marks a release as **unsafe or unwanted for new installs** while leaving it visible for **pins** that already reference it. It is the usual middle ground when something is broken but you must not break historical reproducibility.
- **Full deletion** — Indexes like PyPI **restrict** outright removal because downstream consumers depend on stable URLs and metadata. Expect policy-heavy processes, not a casual button.

### People and ownership

When maintainers change jobs, **transfer** or add **owners/maintainers** in the PyPI web UI *before* access is lost. Documentation and issue templates should list **current contacts**.

### Playbook snippets

- **Broken wheel** — Yank, fix, bump version, upload, announce.
- **README-only typo** — Often a **patch** release if PyPI rendering matters to users.
- **Leaked token** — **Rotate** immediately; only then decide whether a release needs yanking or extra comms.
- **Temptation to reuse a version** — **Don’t**; teach your tooling to always move forward.

The closing code cell walks through short scenario → action reminders; PyPI itself is usually operated through the **website**, not through `pip`.

In [ ]:
scenarios = {
    "broken wheel uploaded": "yank current release and upload a bumped version",
    "typo in README only": "publish a new patch release if the rendered metadata matters",
    "accidentally leaked token": "rotate the token immediately, then assess whether a new release or project cleanup is needed",
    "package moved to new team": "add / transfer maintainers and update documentation",
    "want to reuse the same version number": "do not do it; publish a new version instead",
}

for scenario, action in scenarios.items():
    print(f"- {scenario}: {action}")

print("\nPyPI operations such as yanking, ownership changes, and project deletion are usually performed in the web UI, not via pip itself.")